In [ ]:
import numpy as np
import pandas as pd
from sys import stderr
from numpy import zeros, arange, isscalar,diag, dot,eye, ix_, ones, r_, pi, flatnonzero as find
from scipy.sparse import csr_matrix
from numpy.linalg import solve
import torch

In [23]:
# === Initialization ===
case_name = 'pglib_opf_case3_lmbd'
case_path = f'../excel_outputs/{case_name}.xlsx'
case = pd.read_excel(case_path, sheet_name=['baseMVA','bus','gen','gencost','branch'])

bus_to_idx = {bus: i+1 for i, bus in enumerate(case['bus'].bus_i.values)}
bus_idx = [bus_to_idx[bus] for bus in case['bus'].bus_i.values]
case['bus'].bus_i = case['bus'].bus_i.replace(bus_to_idx) # rename the bus for making PTDF
case['gen'].bus_i = case['gen'].bus_i.replace(bus_to_idx)
case['gencost'].bus_i = case['gencost'].bus_i.replace(bus_to_idx)
case['branch'].bus_i = case['branch'].bus_i.replace(bus_to_idx)
case['branch'].bus_j = case['branch'].bus_j.replace(bus_to_idx)
nbus = case['bus'].shape[0]
ngen = case['gen'].shape[0]
nbranch = case['branch'].shape[0]

# per unit p.u. conversion
baseMVA = case['baseMVA'].values[0][0]
Pmax = case['gen'].Pmax.values / baseMVA
Pmin = case['gen'].Pmin.values / baseMVA
Qmax = case['gen'].Qmax.values / baseMVA
Qmin = case['gen'].Qmin.values / baseMVA
I_max = case['branch'].rateA.values / baseMVA
c2 = case['gencost'].c2.values * baseMVA**2
c1 = case['gencost'].c1.values * baseMVA
c0 = case['gencost'].c0.values

# calculate susceptance, conductance, admittance-square y_sq
# $Z = r + ix$ $Y = g + ib$ $Y = \frac{1}{Z} = \frac{r}{r^2 + x^2} - i\frac{x}{r^2 + x^2}$
# 1. Physics: Admittance Y = g + i*b
r = case['branch']['r'].values
x = case['branch']['x'].values
Z_sq = r**2 + x**2
g = r / Z_sq
b = -x / Z_sq

# 2. Extract Line Charging, Taps, and Phase Shifts
bc = case['branch']['b'].values # MATPOWER branch 'b' is total line charging susceptance
tau = np.where(case['branch']['ratio'].values == 0, 1.0, case['branch']['ratio'].values)
theta_shift = np.radians(case['branch']['angle'].values)

# 3. Extract Shunts from Bus Data
Gs = case['bus']['Gs'].values / baseMVA
Bs = case['bus']['Bs'].values / baseMVA

# State vector dimension D = 2 * |B|
D = 2 * nbus

# Data for QCQP

## 1. Branch Flow Matrices (Equations 14-22)

In [24]:
# Initialize lists to store matrices for all branches
M_pf = []; M_qf = []; M_pt = []; M_qt = []

# Pre-calculate derived branch elements
g11 = g / (tau**2)
g12 = g * np.cos(theta_shift) / tau
g21 = g * np.sin(theta_shift) / tau
g22 = g

b11 = (b + bc/2) / (tau**2)
b12 = b * np.cos(theta_shift) / tau
b21 = b * np.sin(theta_shift) / tau
b22 = b + bc/2

for l in range(nbranch):
    # Python uses 0-based indexing; your dictionary offset to 1-based, so we subtract 1
    i = int(case['branch']['bus_i'].values[l]) - 1
    j = int(case['branch']['bus_j'].values[l]) - 1
    
    # Real and Imaginary indices
    i_B = i + nbus
    j_B = j + nbus

    # --- FROM END ---
    A_pf = np.zeros((D, D))
    A_pf[i, i] = g11[l]
    A_pf[i_B, i_B] = g11[l]
    A_pf[i, j] = -(g12[l] - b21[l])
    A_pf[i_B, j_B] = -(g12[l] - b21[l])
    A_pf[i, j_B] = (g21[l] + b12[l])
    A_pf[i_B, j] = -(g21[l] + b12[l])
    M_pf.append(0.5 * (A_pf + A_pf.T))

    A_qf = np.zeros((D, D))
    A_qf[i, i] = -b11[l]
    A_qf[i_B, i_B] = -b11[l]
    A_qf[i, j] = (b12[l] + g21[l])
    A_qf[i_B, j_B] = (b12[l] + g21[l])
    A_qf[i, j_B] = -(b21[l] - g12[l])
    A_qf[i_B, j] = (b21[l] - g12[l])
    M_qf.append(0.5 * (A_qf + A_qf.T))

    # --- TO END ---
    A_pt = np.zeros((D, D))
    A_pt[j, j] = g22[l]
    A_pt[j_B, j_B] = g22[l]
    A_pt[j, i] = -(g12[l] + b21[l])
    A_pt[j_B, i_B] = -(g12[l] + b21[l])
    A_pt[j, i_B] = -(g21[l] - b12[l])
    A_pt[j_B, i] = (g21[l] - b12[l])
    M_pt.append(0.5 * (A_pt + A_pt.T))

    A_qt = np.zeros((D, D))
    A_qt[j, j] = -b22[l]
    A_qt[j_B, j_B] = -b22[l]
    A_qt[j, i] = (b12[l] - g21[l])
    A_qt[j_B, i_B] = (b12[l] - g21[l])
    A_qt[j, i_B] = (b21[l] + g12[l])
    A_qt[j_B, i] = -(b21[l] + g12[l])
    M_qt.append(0.5 * (A_qt + A_qt.T))

## 2. Nodal Power Injection Matrices (Equations 23-28)

In [25]:
# 1. Build Standard Y_bus matrices (G_bus and B_bus)
G_bus = np.zeros((nbus, nbus))
B_bus = np.zeros((nbus, nbus))

# Add shunts to the diagonals
np.fill_diagonal(G_bus, Gs)
np.fill_diagonal(B_bus, Bs)

# Add branch contributions
for l in range(nbranch):
    i = int(case['branch']['bus_i'].values[l]) - 1
    j = int(case['branch']['bus_j'].values[l]) - 1

    # Diagonal elements
    G_bus[i, i] += g11[l]
    G_bus[j, j] += g22[l]
    B_bus[i, i] += b11[l]
    B_bus[j, j] += b22[l]

    # Off-diagonal elements (symmetric for the line itself, but tap ratios make Y_bus asymmetric)
    G_bus[i, j] += -g12[l]
    G_bus[j, i] += -g12[l] # Assuming simplified model where g12 roughly handles the tap phase
    B_bus[i, j] += -b12[l]
    B_bus[j, i] += -b12[l]

# 2. Build the QCQP Nodal Matrices
M_p = []; M_q = []

for i in range(nbus):
    A_pi = np.zeros((D, D))
    A_qi = np.zeros((D, D))
    
    # Active Power Row mappings (Eq 25)
    A_pi[i, :nbus] = G_bus[i, :]
    A_pi[i, nbus:] = -B_bus[i, :]
    A_pi[i + nbus, :nbus] = B_bus[i, :]
    A_pi[i + nbus, nbus:] = G_bus[i, :]
    M_p.append(0.5 * (A_pi + A_pi.T))
    
    # Reactive Power Row mappings (Eq 27)
    A_qi[i, :nbus] = -B_bus[i, :]
    A_qi[i, nbus:] = -G_bus[i, :]
    A_qi[i + nbus, :nbus] = G_bus[i, :]
    A_qi[i + nbus, nbus:] = -B_bus[i, :]
    M_q.append(0.5 * (A_qi + A_qi.T))

## 3. Branch Angle Difference & Voltage Matrices (Equations 29-34)

In [26]:
M_c = []; M_s = []

for l in range(nbranch):
    i = int(case['branch']['bus_i'].values[l]) - 1
    j = int(case['branch']['bus_j'].values[l]) - 1
    i_B = i + nbus
    j_B = j + nbus

    # Angle Cosine Extraction (Eq 29 & 30)
    A_c = np.zeros((D, D))
    A_c[i, j] = 1
    A_c[i_B, j_B] = 1
    M_c.append(0.5 * (A_c + A_c.T))

    # Angle Sine Extraction (Eq 31 & 32)
    A_s = np.zeros((D, D))
    A_s[i_B, j] = 1
    A_s[i, j_B] = -1
    M_s.append(0.5 * (A_s + A_s.T))

M_V = []
for i in range(nbus):
    # Voltage Magnitude Extraction (Eq 33 & 34)
    A_V = np.zeros((D, D))
    A_V[i, i] = 1
    A_V[i + nbus, i + nbus] = 1
    M_V.append(A_V) # Already symmetric

## 4. Reference Angle Vector

In [27]:
# Identify the slack bus (MATPOWER sets bus type to 3 for slack)
slack_bus_idx = case['bus'][case['bus']['type'] == 3].index[0]

a_ref = np.zeros(D)
# Force the imaginary voltage component of the slack bus to 0
a_ref[slack_bus_idx + nbus] = 1

In [ ]:
def build_G_B_from_branch(n, branch_df, Nb):
    """
    Generates sparse Gn and Bn matrices for a given bus n.
    """
    # Lists to hold the coordinates and values for the sparse matrix
    rows = []
    cols = []
    g_data = []
    b_data = []

    # Speed optimization: Only look at branches connected to bus n
    connected_from = branch_df[branch_df['bus_i'] == n]
    connected_to = branch_df[branch_df['bus_j'] == n]

    # Branches where bus n is the "from" bus (i == n)
    for _, row in connected_from.iterrows():
        j = int(row['bus_j'])
        rows.append(n - 1)
        cols.append(j - 1)
        
        # CORRECTED: Conductance is G, Susceptance is B
        g_data.append(row['conductance'])
        b_data.append(row['susceptance'])

    # Branches where bus n is the "to" bus (j == n)
    for _, row in connected_to.iterrows():
        i = int(row['bus_i'])
        rows.append(n - 1)
        cols.append(i - 1)
        
        # CORRECTED: Conductance is G, Susceptance is B
        g_data.append(row['conductance'])
        b_data.append(row['susceptance'])

    # Build the sparse matrices directly using coordinate format (data, (row_ind, col_ind))
    G_sparse = csr_matrix((g_data, (rows, cols)), shape=(Nb, Nb))
    B_sparse = csr_matrix((b_data, (rows, cols)), shape=(Nb, Nb))

    return G_sparse, B_sparse

def build_Mp_Mq(G, B, n):
    Nb = G.shape[0]
    dim = 2 * Nb

    # We keep Mp and Mq dense because PyTorch will stack these into [ngen, 2Nb, 2Nb] tensors later
    Mp = np.zeros((dim, dim))
    Mq = np.zeros((dim, dim))

    def add_sym(M, i, j, val):
        if i == j:
            M[i, j] += val
        else:
            M[i, j] += val / 2
            M[j, i] += val / 2

    nr = (n - 1)
    ni = (n - 1) + Nb

    # 1. Extract the specific row for bus n (this returns a 1 x Nb sparse matrix)
    G_row = G[nr, :]
    B_row = B[nr, :]

    # 2. Get ONLY the column indices where there is an actual connection (non-zero elements)
    # Since G and B have the same sparsity pattern, we just need the columns from one of them
    _, connected_cols = G_row.nonzero()

    # 3. Iterate only over connected buses
    for k in connected_cols:
        # Extract values efficiently
        Gnk = G_row[0, k]
        Bnk = B_row[0, k]

        kr = k
        ki = k + Nb

        # ----- M^p_n -----
        add_sym(Mp, nr, kr,  Gnk)
        add_sym(Mp, nr, ki, -Bnk)
        add_sym(Mp, ni, ki,  Gnk)
        add_sym(Mp, ni, kr,  Bnk)

        # ----- M^q_n -----
        add_sym(Mq, ni, kr,  Gnk)
        add_sym(Mq, ni, ki, -Bnk)
        add_sym(Mq, nr, ki, -Gnk)
        add_sym(Mq, nr, kr, -Bnk)

    return Mp, Mq

def M_I_single(n, m, y_sq, num_buses):
    """
    Build M^I_nm matrix for a single branch (n, m).
    
    Parameters:
    - n, m: 0-based bus indices (0 to Nb-1)
    - y_sq: Series admittance magnitude squared |y_nm|^2 (scalar)
    - num_buses: total number of buses (Nb)
    """
    dim = 2 * num_buses
    M = np.zeros((dim, dim))

    # ---- Real Part Block ----
    n=n-1
    m=m-1
    M[n, n] += y_sq
    M[m, m] += y_sq
    M[n, m] -= y_sq
    M[m, n] -= y_sq

    # ---- Imaginary Part Block ----
    n_i = n + num_buses
    m_i = m + num_buses

    M[n_i, n_i] += y_sq
    M[m_i, m_i] += y_sq
    M[n_i, m_i] -= y_sq
    M[m_i, n_i] -= y_sq

    return M

def M_v_single(n, num_buses):
    dim = 2 * num_buses
    M = np.zeros((dim, dim))

    # Convert 1-based 'n' to 0-based Python index
    n_idx = n - 1 
    M[n_idx, n_idx] = 1

    n_i = num_buses + n_idx
    M[n_i, n_i] = 1 
    return M

In [31]:
# def build_M_pf_qf_single(i, k, g, b, g_sh, b_sh, T, phi, num_buses):
#     """
#     Builds the M_P_ik and M_Q_ik quadratic constraint matrices for a single branch.
    
#     Parameters:
#     - i, k: 0-based bus indices for the 'from' and 'to' buses
#     - g, b: Series conductance and susceptance of the branch
#     - g_sh, b_sh: Shunt conductance and susceptance of the branch
#     - T: Transformer tap ratio (use 1.0 for standard transmission lines)
#     - phi: Transformer phase shift IN RADIANS (use 0.0 for standard lines)
#     - num_buses: Total number of buses (N_b)
    
#     Returns:
#     - M_P: (2Nb x 2Nb) numpy array for Active Power flow (P_ik)
#     - M_Q: (2Nb x 2Nb) numpy array for Reactive Power flow (Q_ik)
#     """
#     dim = 2 * num_buses
#     M_P = np.zeros((dim, dim))
#     M_Q = np.zeros((dim, dim))

#     def add_sym(M, row, col, val):
#         """Helper to safely build symmetric quadratic matrices"""
#         if row == col:
#             M[row, col] += val
#         else:
#             M[row, col] += val / 2.0
#             M[col, row] += val / 2.0

#     # Index offsets for imaginary parts (F variables)
#     i_imag = i + num_buses
#     k_imag = k + num_buses

#     # Precompute trigonometry and tap ratio terms for efficiency
#     cos_phi = np.cos(phi)
#     sin_phi = np.np.sin(phi)
#     T_sq = T ** 2

#     # ==========================================
#     # 1. M_P Matrix (Active Power P_ik)
#     # ==========================================
    
#     # Term 1: (E_i^2 + F_i^2) coefficients
#     coeff_P_ii = (1.0 / T_sq) * (g + g_sh / 2.0)
#     add_sym(M_P, i, i, coeff_P_ii)             # E_i^2
#     add_sym(M_P, i_imag, i_imag, coeff_P_ii)   # F_i^2
    
#     # Term 2: (E_i E_k + F_i F_k) coefficients
#     coeff_P_ik_real = (1.0 / T) * (-g * cos_phi + b * sin_phi)
#     add_sym(M_P, i, k, coeff_P_ik_real)             # E_i E_k
#     add_sym(M_P, i_imag, k_imag, coeff_P_ik_real)   # F_i F_k
    
#     # Term 3: (E_i F_k - F_i E_k) coefficients
#     coeff_P_ik_imag = (1.0 / T) * (g * sin_phi + b * cos_phi)
#     add_sym(M_P, i, k_imag, coeff_P_ik_imag)        # E_i F_k
#     add_sym(M_P, i_imag, k, -coeff_P_ik_imag)       # -F_i E_k (Note the negative sign!)

#     # ==========================================
#     # 2. M_Q Matrix (Reactive Power Q_ik)
#     # ==========================================
    
#     # Term 1: (E_i^2 + F_i^2) coefficients
#     coeff_Q_ii = -(1.0 / T_sq) * (b + b_sh / 2.0)
#     add_sym(M_Q, i, i, coeff_Q_ii)             # E_i^2
#     add_sym(M_Q, i_imag, i_imag, coeff_Q_ii)   # F_i^2
    
#     # Term 2: (E_i E_k + F_i F_k) coefficients
#     coeff_Q_ik_real = (1.0 / T) * (g * sin_phi + b * cos_phi)
#     add_sym(M_Q, i, k, coeff_Q_ik_real)             # E_i E_k
#     add_sym(M_Q, i_imag, k_imag, coeff_Q_ik_real)   # F_i F_k
    
#     # Term 3: (E_i F_k - F_i E_k) coefficients
#     coeff_Q_ik_imag = (1.0 / T) * (g * cos_phi - b * sin_phi)
#     add_sym(M_Q, i, k_imag, coeff_Q_ik_imag)        # E_i F_k
#     add_sym(M_Q, i_imag, k, -coeff_Q_ik_imag)       # -F_i E_k (Note the negative sign!)

#     return M_P, M_Q


In [ ]:
G_n, B_n = {}, {}
Mp_ng, Mq_ng = {}, {}

for n in case['gen']['bus_i']:
    G_n[n], B_n[n] = build_G_B_from_branch(n, case['branch'], nbus)
    Mp_ng[n], Mq_ng[n] = build_Mp_Mq(G_n[n], B_n[n], n)

G_n, B_n = {}, {}
Mp_nb, Mq_nb = {}, {}

for n in case['bus']['bus_i']:
    G_n[n], B_n[n] = build_G_B_from_branch(n, case['branch'], nbus)
    Mp_nb[n], Mq_nb[n] = build_Mp_Mq(G_n[n], B_n[n], n)

Mv_n = {}
for n in case['bus']['bus_i']:
    Mv_n[n]=M_v_single(n, nbus)

MI_nm={}
for line in case['branch'].index:
    n, m = case['branch'].loc[line,['bus_i','bus_j']]
    n, m = int(n), int(m)
    y_sq = case['branch'].loc[line,"y_sq"]
    MI_nm[(n, m)]=M_I_single(n, m, y_sq, nbus)


In [119]:
# Assuming nbus is defined earlier
Ms = np.zeros((2*nbus, 2*nbus))
Ms[nbus, nbus] = 1

Mc = np.zeros((2*nbus, 2*nbus))
# Find the number of generators to properly slice the gencost dataframe
ngen = len(case['gen'])
ncost = len(case['gencost'])
# Extract generator bus IDs
gen_buses = case['gen']['bus_i'].values
# Extract Active Power (P) costs (always the first 'ngen' rows)
P_costs = case['gencost']['c1'].values[:ngen]
cp = dict(zip(gen_buses, P_costs))

# Check for Reactive Power (Q) costs based on MATPOWER standard
if ncost == 2 * ngen:
    # Extract Reactive Power (Q) costs (the next 'ngen' rows)
    Q_costs = case['gencost']['c1'].values[ngen : 2*ngen]
    cq = dict(zip(gen_buses, Q_costs))
else:
    # No Q costs provided, set all to 0
    cq = dict(zip(gen_buses, np.zeros(ngen)))

# Build the final Mc matrix
for n in gen_buses:
    Mc += cp[n] * Mp_ng[n] + cq[n] * Mq_ng[n]
    
cost = case['gencost']['c1'].values
# Scale power limits and demands by baseMVA to convert to per-unit
pmax = case['gen']['Pmax'].values / baseMVA
pmin = case['gen']['Pmin'].values / baseMVA

qmax = case['gen']['Qmax'].values / baseMVA
qmin = case['gen']['Qmin'].values / baseMVA

Pd = case['bus']['Pd'].values / baseMVA
Qd = case['bus']['Qd'].values / baseMVA

# FIXED: Voltage limits are already in per-unit! Do NOT divide by baseMVA.
Vmax = case['bus']['Vmax'].values 
Vmin = case['bus']['Vmin'].values 

Pd_gen, Qd_gen = [], []
for b in gen_buses:
    msk = case['bus']['bus_i'] == b
    # Extracting scalar values using [0] to avoid appending arrays of arrays
    Pd_gen.append(Pd[msk][0] if any(msk) else 0.0) 
    Qd_gen.append(Qd[msk][0] if any(msk) else 0.0)


In [120]:
Ms = np.zeros((2*nbus,2*nbus))
Ms[nbus,nbus] = 1

Mc = np.zeros((2*nbus,2*nbus))
cp = dict(zip(case['gencost'].bus_i.values,case['gencost'].c1.values))

cq = dict(zip(case['gencost'].bus_i.values,case['gencost'].c1.values))

for n in case['gen'].bus_i:
    Mc += cp[n] * Mp_ng[n] + cq[n] * Mq_ng[n]
    
cost = case['gencost'].c1.values

pmax = case['gen'].Pmax.values / baseMVA
pmin = case['gen'].Pmin.values / baseMVA

qmax = case['gen'].Qmax.values / baseMVA
qmin = case['gen'].Qmin.values / baseMVA

Pd = case['bus'].Pd.values / baseMVA
Qd = case['bus'].Qd.values / baseMVA

Vmax = case['bus'].Vmax.values / baseMVA
Vmin = case['bus'].Vmin.values / baseMVA

Pd_gen, Qd_gen = [], []
for b in case['gen']['bus_i']:
    msk = case['bus']['bus_i']==b
    Pd_gen.append(case['bus'].loc[msk, 'Pd'].values)
    Qd_gen.append(case['bus'].loc[msk, 'Qd'].values)    

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32

# ------------------------------------------------------------
# 1) Ordered bus sets
# ------------------------------------------------------------
all_bus_ids = [int(n) for n in case['bus']['bus_i']]

# unique generator buses only (important for bus-based constraints)
gen_bus_ids = sorted(set(int(n) for n in case['gen']['bus_i']))

# buses in N_b \ N_g
load_bus_ids = [b for b in all_bus_ids if b not in set(gen_bus_ids)]

# ordered branch keys matching your M_I construction
branch_ids = [
    (int(case['branch'].loc[line, 'bus_i']), int(case['branch'].loc[line, 'bus_j']))
    for line in case['branch'].index
]

# map bus ID -> position in full bus vector Pd/Qd
bus_id_to_pos = {bus_id: i for i, bus_id in enumerate(all_bus_ids)}

nbus = len(all_bus_ids)
ngen = len(gen_bus_ids)       # number of generator buses
nload = len(load_bus_ids)     # number of non-generator buses
nbranch = len(branch_ids)
d = 2 * nbus

# ------------------------------------------------------------
# 2) Stack quadratic matrices in the correct order
# ------------------------------------------------------------
M_p_gen_stack = torch.stack(
    [torch.as_tensor(Mp_ng[b], dtype=dtype, device=device) for b in gen_bus_ids],
    dim=0
)   # [ngen, d, d]

M_q_gen_stack = torch.stack(
    [torch.as_tensor(Mq_ng[b], dtype=dtype, device=device) for b in gen_bus_ids],
    dim=0
)   # [ngen, d, d]

M_p_load_stack = torch.stack(
    [torch.as_tensor(Mp_nb[b], dtype=dtype, device=device) for b in load_bus_ids],
    dim=0
)   # [nload, d, d]

M_q_load_stack = torch.stack(
    [torch.as_tensor(Mq_nb[b], dtype=dtype, device=device) for b in load_bus_ids],
    dim=0
)   # [nload, d, d]

M_v_stack = torch.stack(
    [torch.as_tensor(Mv_n[b], dtype=dtype, device=device) for b in all_bus_ids],
    dim=0
)   # [nbus, d, d]

M_I_stack = torch.stack(
    [torch.as_tensor(MI_nm[key], dtype=dtype, device=device) for key in branch_ids],
    dim=0
)   # [nbranch, d, d]

M_c = torch.as_tensor(Mc, dtype=dtype, device=device)      # [d, d]
M_s = torch.as_tensor(Ms, dtype=dtype, device=device)      # [d, d]

# ------------------------------------------------------------
# 3) Symmetrize matrices (recommended)
# ------------------------------------------------------------
M_c = 0.5 * (M_c + M_c.T)
M_p_gen_stack = 0.5 * (M_p_gen_stack + M_p_gen_stack.transpose(-1, -2))
M_q_gen_stack = 0.5 * (M_q_gen_stack + M_q_gen_stack.transpose(-1, -2))
M_p_load_stack = 0.5 * (M_p_load_stack + M_p_load_stack.transpose(-1, -2))
M_q_load_stack = 0.5 * (M_q_load_stack + M_q_load_stack.transpose(-1, -2))
M_v_stack = 0.5 * (M_v_stack + M_v_stack.transpose(-1, -2))
M_I_stack = 0.5 * (M_I_stack + M_I_stack.transpose(-1, -2))
# M_s usually does not need symmetrization here, but harmless if desired:
# M_s = 0.5 * (M_s + M_s.T)

# ------------------------------------------------------------
# 4) Bus-level demand vectors
# ------------------------------------------------------------
Pd_bus = np.asarray(case['bus'].Pd.values, dtype=np.float32) / baseMVA
Qd_bus = np.asarray(case['bus'].Qd.values, dtype=np.float32) / baseMVA

# extract in the same order as the matrix stacks
Pd_load = np.asarray([Pd_bus[bus_id_to_pos[b]] for b in load_bus_ids], dtype=np.float32)
Qd_load = np.asarray([Qd_bus[bus_id_to_pos[b]] for b in load_bus_ids], dtype=np.float32)

Pd_gen = np.asarray([Pd_bus[bus_id_to_pos[b]] for b in gen_bus_ids], dtype=np.float32)
Qd_gen = np.asarray([Qd_bus[bus_id_to_pos[b]] for b in gen_bus_ids], dtype=np.float32)

# ------------------------------------------------------------
# 5) Generator bounds aggregated per generator bus
#    (important if multiple generators are connected to one bus)
# ------------------------------------------------------------
pmax_by_bus, pmin_by_bus = {}, {}
qmax_by_bus, qmin_by_bus = {}, {}

for _, row in case['gen'].iterrows():
    b = int(row['bus_i'])
    pmax_by_bus[b] = pmax_by_bus.get(b, 0.0) + float(row['Pmax']) / baseMVA
    pmin_by_bus[b] = pmin_by_bus.get(b, 0.0) + float(row['Pmin']) / baseMVA
    qmax_by_bus[b] = qmax_by_bus.get(b, 0.0) + float(row['Qmax']) / baseMVA
    qmin_by_bus[b] = qmin_by_bus.get(b, 0.0) + float(row['Qmin']) / baseMVA

pmax_g = np.asarray([pmax_by_bus[b] for b in gen_bus_ids], dtype=np.float32)
pmin_g = np.asarray([pmin_by_bus[b] for b in gen_bus_ids], dtype=np.float32)
qmax_g = np.asarray([qmax_by_bus[b] for b in gen_bus_ids], dtype=np.float32)
qmin_g = np.asarray([qmin_by_bus[b] for b in gen_bus_ids], dtype=np.float32)

# ------------------------------------------------------------
# 6) Voltage / branch limits
# ------------------------------------------------------------
# keep exactly your scaling convention
Vmax_arr = np.asarray(case['bus'].Vmax.values, dtype=np.float32) / baseMVA
Vmin_arr = np.asarray(case['bus'].Vmin.values, dtype=np.float32) / baseMVA
Imax_arr = np.asarray(case['branch'].I_max.values, dtype=np.float32)

# ------------------------------------------------------------
# 7) Tensor index positions for extracting batched Pd/Qd inside loss
# ------------------------------------------------------------
load_bus_pos = torch.tensor(
    [bus_id_to_pos[b] for b in load_bus_ids],
    dtype=torch.long,
    device=device
)

gen_bus_pos = torch.tensor(
    [bus_id_to_pos[b] for b in gen_bus_ids],
    dtype=torch.long,
    device=device
)

# ------------------------------------------------------------
# 8) kappa
# ------------------------------------------------------------
kappa_value = 0.0
if 'kappa' in locals():
    kappa_value = float(kappa)

# ------------------------------------------------------------
# 9) Final problem dictionary for the PINN loss
# ------------------------------------------------------------
problem = {
    "M_c": M_c,
    "M_p_load": M_p_load_stack,
    "M_q_load": M_q_load_stack,
    "M_p_gen": M_p_gen_stack,
    "M_q_gen": M_q_gen_stack,
    "M_v": M_v_stack,
    "M_I": M_I_stack,
    "M_s": M_s,

    # base vectors (useful for reference/debugging)
    "Pd_bus": torch.as_tensor(Pd_bus, dtype=dtype, device=device),
    "Qd_bus": torch.as_tensor(Qd_bus, dtype=dtype, device=device),
    "Pd_load": torch.as_tensor(Pd_load, dtype=dtype, device=device),
    "Qd_load": torch.as_tensor(Qd_load, dtype=dtype, device=device),
    "Pd_gen": torch.as_tensor(Pd_gen, dtype=dtype, device=device),
    "Qd_gen": torch.as_tensor(Qd_gen, dtype=dtype, device=device),

    "pmax_g": torch.as_tensor(pmax_g, dtype=dtype, device=device),
    "pmin_g": torch.as_tensor(pmin_g, dtype=dtype, device=device),
    "qmax_g": torch.as_tensor(qmax_g, dtype=dtype, device=device),
    "qmin_g": torch.as_tensor(qmin_g, dtype=dtype, device=device),

    "Vmax": torch.as_tensor(Vmax_arr, dtype=dtype, device=device),
    "Vmin": torch.as_tensor(Vmin_arr, dtype=dtype, device=device),
    "Imax": torch.as_tensor(Imax_arr, dtype=dtype, device=device),

    # use these inside compute_loss_aug_lagrangian:
    # Pd_load_batch = Pd_batch[:, problem["load_bus_pos"]]
    # Pd_gen_batch  = Pd_batch[:, problem["gen_bus_pos"]]
    "load_bus_pos": load_bus_pos,
    "gen_bus_pos": gen_bus_pos,

    "kappa": torch.as_tensor(kappa_value, dtype=dtype, device=device),

    # metadata
    "all_bus_ids": all_bus_ids,
    "gen_bus_ids": gen_bus_ids,
    "load_bus_ids": load_bus_ids,
    "branch_ids": branch_ids,
}

print("Constructed PINN problem data:")
print(f"  nbus   = {nbus}")
print(f"  ngen   = {ngen}   (generator buses)")
print(f"  nload  = {nload}  (non-generator buses)")
print(f"  nbranch= {nbranch}")
print(f"  M_p_gen shape  = {tuple(problem['M_p_gen'].shape)}")
print(f"  M_p_load shape = {tuple(problem['M_p_load'].shape)}")
print(f"  M_v shape      = {tuple(problem['M_v'].shape)}")
print(f"  M_I shape      = {tuple(problem['M_I'].shape)}")
print(f"  load_bus_pos   = {problem['load_bus_pos'].tolist()}")
print(f"  gen_bus_pos    = {problem['gen_bus_pos'].tolist()}")

AttributeError: 'DataFrame' object has no attribute 'I_max'

In [56]:
import torch

def gaussian_batch(base_tensor, batch_size, variation_std=0.05, clamp_min=0.0):
    """
    Create a batch of tensors with Gaussian random variations.
    
    Args:
        base_tensor (torch.Tensor): original tensor of shape [N]
        batch_size (int): number of samples in batch
        variation_std (float): standard deviation of Gaussian noise, relative to base_tensor
        clamp_min (float): minimum allowed value for perturbed tensor
    
    Returns:
        torch.Tensor: batch of shape [batch_size, N]
    """
    # Expand base tensor to batch
    base_batch = base_tensor.unsqueeze(0).repeat(batch_size, 1)
    
    # Gaussian noise scaled by base_tensor
    noise = variation_std * base_tensor.unsqueeze(0) * torch.randn_like(base_batch)
    
    # Perturbed batch
    batch = base_batch + noise
    
    # Clamp if necessary
    if clamp_min is not None:
        batch = torch.clamp(batch, min=clamp_min)
    
    return batch


def uniform_batch(base_tensor, batch_size, variation=0.05, clamp_min=0.0):
    """
    Create a batch of tensors with uniform random variations.
    
    Args:
        base_tensor (torch.Tensor): original tensor of shape [N]
        batch_size (int): number of samples in batch
        variation (float): maximum relative deviation (±variation)
        clamp_min (float): minimum allowed value for perturbed tensor
    
    Returns:
        torch.Tensor: batch of shape [batch_size, N]
    """
    base_batch = base_tensor.unsqueeze(0).repeat(batch_size, 1)
    
    # Uniform noise in [-variation, +variation]
    noise = (2 * torch.rand_like(base_batch) - 1) * variation
    
    batch = base_batch * (1 + noise)
    
    if clamp_min is not None:
        batch = torch.clamp(batch, min=clamp_min)
    
    return batch

In [57]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def quad_batch(v: torch.Tensor, M: torch.Tensor) -> torch.Tensor:
    # v: [B, d], M: [d, d] -> [B]
    return torch.einsum("bi,ij,bj->b", v, M, v)

def quad_batch_stack(v: torch.Tensor, M: torch.Tensor) -> torch.Tensor:
    # v: [B, d], M: [K, d, d] -> [B, K]
    return torch.einsum("bi,kij,bj->bk", v, M, v)

def gaussian_batch(x: torch.Tensor, batch_size: int, variation_std: float = 0.05) -> torch.Tensor:
    # x: [n] -> [B, n]
    x = x.reshape(1, -1).expand(batch_size, -1)
    scale = torch.clamp(x.abs(), min=1.0)
    noise = variation_std * torch.randn_like(x) * scale
    return x + noise

def to_row_batch(x: torch.Tensor, B: int) -> torch.Tensor:
    # robust expansion for vectors that may be [N], [N,1], [1,N], etc.
    return x.reshape(1, -1).expand(B, -1)


# ------------------------------------------------------------
# Verification-oriented feasibility network
# ------------------------------------------------------------

class FeasibleVoltageMLP(nn.Module):
    """
    Input:
        Pd: [B, nbus]
        Qd: [B, nbus]

    Output:
        v:  [B, 2*nbus]

    Architecture choices for verification-readiness:
      - only affine + ReLU + clamp
      - no dual variables
      - no implicit optimization layer
      - no eig/SVD/nullspace operations
      - slack imaginary part fixed to zero by construction
    """
    def __init__(
        self,
        nbus: int,
        hidden1: int = 256,
        hidden2: int = 256,
        hidden3: int = 256,
        slack_imag_idx: int = None,
        bound_margin: float = 0.98,   # keep outputs slightly inside box
    ):
        super().__init__()
        self.nbus = nbus
        self.in_dim = 2 * nbus
        self.out_dim = 2 * nbus
        self.slack_imag_idx = nbus if slack_imag_idx is None else int(slack_imag_idx)
        self.bound_margin = float(bound_margin)

        self.net = nn.Sequential(
            nn.Linear(self.in_dim, hidden1),
            nn.ReLU(),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Linear(hidden2, hidden3),
            nn.ReLU(),
            nn.Linear(hidden3, self.out_dim),
        )

    def forward(self, Pd: torch.Tensor, Qd: torch.Tensor, Vmax: torch.Tensor) -> torch.Tensor:
        """
        Vmax: [nbus] or [B, nbus]
        Returns v = [vr, vi] with each component bounded in [-bound_margin*Vmax, +bound_margin*Vmax]
        and slack imaginary component exactly zero.
        """
        B = Pd.shape[0]
        x = torch.cat([Pd, Qd], dim=-1)            # [B, 2*nbus]
        raw = self.net(x)                          # [B, 2*nbus]

        vr_raw = raw[:, :self.nbus]                # [B, nbus]
        vi_raw = raw[:, self.nbus:]                # [B, nbus]

        Vmax_b = Vmax.reshape(1, -1).expand(B, -1) if Vmax.dim() == 1 else Vmax
        comp_bound = self.bound_margin * Vmax_b

        # Piecewise-linear bounded map: clamp is simpler than tanh/sigmoid for verification-oriented use
        vr = torch.clamp(vr_raw, min=-comp_bound, max=comp_bound)
        vi = torch.clamp(vi_raw, min=-comp_bound, max=comp_bound)

        # Enforce slack imaginary part = 0 exactly
        vi = vi.clone()
        vi[:, self.slack_imag_idx - self.nbus if self.slack_imag_idx >= self.nbus else 0] = 0.0
        # In your previous Ms, Ms[nbus, nbus] = 1, so slack_imag_idx = nbus means first imag entry.
        # The expression above converts the global index into the local imag-block index.

        v = torch.cat([vr, vi], dim=-1)            # [B, 2*nbus]
        return v


# ------------------------------------------------------------
# Pure feasibility loss
# ------------------------------------------------------------

def compute_feasibility_loss(
    model: nn.Module,
    Pd_batch: torch.Tensor,
    Qd_batch: torch.Tensor,
    problem: dict,
    w_eq_p: float = 10.0,
    w_eq_q: float = 10.0,
    w_ineq_gen: float = 5.0,
    w_ineq_v: float = 5.0,
    w_ineq_I: float = 5.0,
    w_slack: float = 20.0,
    w_obj: float = 0.01,
):
    B = Pd_batch.shape[0]
    device = Pd_batch.device

    # --------------------------------------------------------
    # pull problem data
    # --------------------------------------------------------
    M_c      = problem["M_c"]          # [d,d]
    M_p_load = problem["M_p_load"]     # [nload,d,d]
    M_q_load = problem["M_q_load"]     # [nload,d,d]
    M_p_gen  = problem["M_p_gen"]      # [ngen,d,d]
    M_q_gen  = problem["M_q_gen"]      # [ngen,d,d]
    M_v      = problem["M_v"]          # [nbus,d,d]
    M_I      = problem["M_I"]          # [nbranch,d,d]
    M_s      = problem["M_s"]          # [d,d]

    load_bus_pos = problem["load_bus_pos"]   # [nload]
    gen_bus_pos  = problem["gen_bus_pos"]    # [ngen]

    pmax_g = to_row_batch(problem["pmax_g"], B)   # [B,ngen]
    pmin_g = to_row_batch(problem["pmin_g"], B)
    qmax_g = to_row_batch(problem["qmax_g"], B)
    qmin_g = to_row_batch(problem["qmin_g"], B)
    Vmax   = to_row_batch(problem["Vmax"], B)     # [B,nbus]
    Vmin   = to_row_batch(problem["Vmin"], B)
    Imax   = to_row_batch(problem["Imax"], B)     # [B,nbranch]

    # --------------------------------------------------------
    # sample-dependent demand values
    # --------------------------------------------------------
    Pd_load = Pd_batch[:, load_bus_pos]      # [B,nload]
    Qd_load = Qd_batch[:, load_bus_pos]      # [B,nload]
    Pd_gen  = Pd_batch[:, gen_bus_pos]       # [B,ngen]
    Qd_gen  = Qd_batch[:, gen_bus_pos]       # [B,ngen]

    # --------------------------------------------------------
    # predict primal voltage
    # --------------------------------------------------------
    v = model(Pd_batch, Qd_batch, Vmax=problem["Vmax"].reshape(-1))

    # --------------------------------------------------------
    # constraint values
    # --------------------------------------------------------
    vp_load = quad_batch_stack(v, M_p_load)   # [B,nload]
    vq_load = quad_batch_stack(v, M_q_load)   # [B,nload]
    vp_gen  = quad_batch_stack(v, M_p_gen)    # [B,ngen]
    vq_gen  = quad_batch_stack(v, M_q_gen)    # [B,ngen]
    vv      = quad_batch_stack(v, M_v)        # [B,nbus]
    vI      = quad_batch_stack(v, M_I)        # [B,nbranch]
    vs      = quad_batch(v, M_s).unsqueeze(-1)

    # Equalities on non-generator buses
    h_pb = vp_load + Pd_load
    h_qb = vq_load + Qd_load

    # Generator inequalities
    g_pg_u = vp_gen - (pmax_g - Pd_gen)
    g_pg_l = -vp_gen + (pmin_g - Pd_gen)
    g_qg_u = vq_gen - (qmax_g - Qd_gen)
    g_qg_l = -vq_gen + (qmin_g - Qd_gen)

    # Voltage / current inequalities
    g_v_u = vv - Vmax.pow(2)
    g_v_l = -vv + Vmin.pow(2)
    g_I   = vI - Imax.pow(2)

    # Slack equality
    h_s = vs

    # --------------------------------------------------------
    # residual losses
    # --------------------------------------------------------
    loss_eq_p = h_pb.pow(2).mean()
    loss_eq_q = h_qb.pow(2).mean()

    loss_gen_ineq = (
        F.relu(g_pg_u).pow(2).mean()
        + F.relu(g_pg_l).pow(2).mean()
        + F.relu(g_qg_u).pow(2).mean()
        + F.relu(g_qg_l).pow(2).mean()
    )

    loss_v_ineq = (
        F.relu(g_v_u).pow(2).mean()
        + F.relu(g_v_l).pow(2).mean()
    )

    loss_I_ineq = F.relu(g_I).pow(2).mean()

    # h_s should already be zero by construction if Ms selects the slack imaginary component,
    # but keep this term as a safety check.
    loss_slack = h_s.pow(2).mean()

    # small objective regularizer (optional, to bias toward cheaper feasible points)
    obj = quad_batch(v, M_c).mean()

    total_loss = (
        w_eq_p * loss_eq_p
        + w_eq_q * loss_eq_q
        + w_ineq_gen * loss_gen_ineq
        + w_ineq_v * loss_v_ineq
        + w_ineq_I * loss_I_ineq
        + w_slack * loss_slack
        + w_obj * obj
    )

    diagnostics = {
        "loss_total": total_loss.detach(),
        "loss_eq_p": loss_eq_p.detach(),
        "loss_eq_q": loss_eq_q.detach(),
        "loss_gen_ineq": loss_gen_ineq.detach(),
        "loss_v_ineq": loss_v_ineq.detach(),
        "loss_I_ineq": loss_I_ineq.detach(),
        "loss_slack": loss_slack.detach(),
        "obj": obj.detach(),
        "max_abs_h_pb": h_pb.abs().max().detach(),
        "max_abs_h_qb": h_qb.abs().max().detach(),
        "max_g_pg_u": g_pg_u.max().detach(),
        "max_g_pg_l": g_pg_l.max().detach(),
        "max_g_qg_u": g_qg_u.max().detach(),
        "max_g_qg_l": g_qg_l.max().detach(),
        "max_g_v_u": g_v_u.max().detach(),
        "max_g_v_l": g_v_l.max().detach(),
        "max_g_I": g_I.max().detach(),
    }

    return total_loss, diagnostics


# ------------------------------------------------------------
# Build model
# ------------------------------------------------------------

device = problem["M_c"].device
nbus = len(problem["all_bus_ids"])

# In your previous construction Ms[nbus, nbus] = 1, so global slack imag index is nbus
slack_imag_idx = nbus

model = FeasibleVoltageMLP(
    nbus=nbus,
    hidden1=10*nbus,
    hidden2=5*nbus,
    hidden3=10*nbus,
    slack_imag_idx=slack_imag_idx,
    bound_margin=0.98,
).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1000, gamma=0.5)

Pd_bus_base = problem["Pd_bus"].reshape(-1)
Qd_bus_base = problem["Qd_bus"].reshape(-1)




In [58]:
# ------------------------------------------------------------
# Training loop
# ------------------------------------------------------------

num_epochs = 3000
batch_size = 32
best_loss = float("inf")
best_state = None

for epoch in range(num_epochs):
    # sample demand scenarios
    Pd_batch = gaussian_batch(Pd_bus_base, batch_size=batch_size, variation_std=0.05)
    Qd_batch = gaussian_batch(Qd_bus_base, batch_size=batch_size, variation_std=0.05)

    # demands should remain nonnegative
    Pd_batch = torch.clamp(Pd_batch, min=0.0)
    Qd_batch = torch.clamp(Qd_batch, min=0.0)

    optimizer.zero_grad()

    loss, diagnostics = compute_feasibility_loss(
        model=model,
        Pd_batch=Pd_batch,
        Qd_batch=Qd_batch,
        problem=problem,
        w_eq_p=10.0,
        w_eq_q=10.0,
        w_ineq_gen=5.0,
        w_ineq_v=5.0,
        w_ineq_I=5.0,
        w_slack=20.0,
        w_obj=0.01,
    )

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
    optimizer.step()
    scheduler.step()

    loss_value = loss.detach().item()
    if loss_value < best_loss:
        best_loss = loss_value
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if epoch % 100 == 0:
        print(f"\nEpoch {epoch:5d} | loss = {loss_value:.6e}")
        print({k: v.detach().item() for k, v in diagnostics.items()})

# restore best model
if best_state is not None:
    model.load_state_dict(best_state)

print("\nBest training loss:", best_loss)





Epoch     0 | loss = 2.054241e+02
{'loss_total': 205.42413330078125, 'loss_eq_p': 18.244476318359375, 'loss_eq_q': 2.292607545852661, 'loss_gen_ineq': 0.006596775725483894, 'loss_v_ineq': 1.1064282823269878e-08, 'loss_I_ineq': 0.0, 'loss_slack': 0.0, 'obj': 2.030914545059204, 'max_abs_h_pb': 22.389909744262695, 'max_abs_h_qb': 5.310649871826172, 'max_g_pg_u': -3.711991786956787, 'max_g_pg_l': 0.1108459010720253, 'max_g_qg_u': -3.16375470161438, 'max_g_qg_l': -2.246126413345337, 'max_g_v_u': 0.00011141680442960933, 'max_g_v_l': -3.52084098267369e-05, 'max_g_I': -1291322.375}

Epoch   100 | loss = 2.060926e+02
{'loss_total': 206.09263610839844, 'loss_eq_p': 18.309724807739258, 'loss_eq_q': 2.297386884689331, 'loss_gen_ineq': 0.00825719628483057, 'loss_v_ineq': 1.2206811561554787e-08, 'loss_I_ineq': 0.0, 'loss_slack': 0.0, 'obj': -1.9756052494049072, 'max_abs_h_pb': 22.377777099609375, 'max_abs_h_qb': 5.425243377685547, 'max_g_pg_u': -3.663360834121704, 'max_g_pg_l': 0.1697320193052292, 

In [43]:
# ------------------------------------------------------------
# Final evaluation on one fresh batch
# ------------------------------------------------------------

with torch.no_grad():
    Pd_test = gaussian_batch(Pd_bus_base, batch_size=64, variation_std=0.05)
    Qd_test = gaussian_batch(Qd_bus_base, batch_size=64, variation_std=0.05)
    Pd_test = torch.clamp(Pd_test, min=0.0)
    Qd_test = torch.clamp(Qd_test, min=0.0)

    test_loss, test_diag = compute_feasibility_loss(
        model=model,
        Pd_batch=Pd_test,
        Qd_batch=Qd_test,
        problem=problem,
        w_eq_p=10.0,
        w_eq_q=10.0,
        w_ineq_gen=5.0,
        w_ineq_v=5.0,
        w_ineq_I=5.0,
        w_slack=20.0,
        w_obj=0.01,
    )

print("\nFinal test diagnostics:")
print({k: v.detach().item() for k, v in test_diag.items()})

# The trained object you want for later verification is `model`.
# A verification pipeline can then check, over an input demand set,
# whether the resulting v satisfies:
#   h_pb = 0, h_qb = 0,
#   g_pg_u <= 0, g_pg_l <= 0,
#   g_qg_u <= 0, g_qg_l <= 0,
#   g_v_u <= 0, g_v_l <= 0,
#   g_I <= 0,
#   h_s = 0.


Final test diagnostics:
{'loss_total': 204.63502502441406, 'loss_eq_p': 18.165790557861328, 'loss_eq_q': 2.294523239135742, 'loss_gen_ineq': 0.007044778671115637, 'loss_v_ineq': 1.2206812449733206e-08, 'loss_I_ineq': 0.0, 'loss_slack': 0.0, 'obj': -0.3332103192806244, 'max_abs_h_pb': 21.82909393310547, 'max_abs_h_qb': 5.523430824279785, 'max_g_pg_u': -3.662353277206421, 'max_g_pg_l': 0.11239556223154068, 'max_g_qg_u': -3.145278215408325, 'max_g_qg_l': -2.25, 'max_g_v_u': 0.00011141680442960933, 'max_g_v_l': -3.52084098267369e-05, 'max_g_I': -430440.75}
